In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

TODO:

- Try class for `Card` later? `namedtuple` works for now.
- Explore ways to design `Player` and `Round` classes.
- Try `IntEnum` for Poker hand rankings. Currently, we use integer values from 14 to 1 to indicate the poker hand rankings. Using `IntEnum` may make it more readable. Read more about enum: https://docs.python.org/3/library/enum.html
- Try passing `Round` to `Player` when taking action to update `stage_bet`.
- Player should not be able to check if `stage_bet != player.bet`.
- Add possible actions for players.
- **Odd chips vanish** — `self.pot // len(winners)` drops the remainder
- Implement the rule: If everyone folds except for one player, that player wins
- Try rewriting `evaluate_hand` lazily.
- Implement all-in
- Add pyskills
- Rather than printing, maybe add note messages?
- Add GUI with fasthtml
- Deploy on server

In [ ]:
from fastcore.utils import *
from fastcore.test import *

import random
from collections import namedtuple, Counter
from enum import Enum, auto
from dataclasses import dataclass, field
from typing import Callable

# core

> Texas hold'em game logic 

In this dialog, I want to go through the process of writing a poker program myself but following the approach I learned in the cs212 class. My goal is not to copy everything from the lesson exactly, but follow the steps: understanding the problem, specifying, concept inventory, etc. that I can apply to solving any problem. To challenge myself, I will write texas hold'em poker.

Here's the process Norvig taught:

**1. Understand** — Start with a vague understanding and refine it. Make an inventory of all the concepts you'll need to deal with (data types, operations, etc.).

**2. Specify** — Define *what* needs to happen: inputs, outputs, what each function should accomplish. Think of it as a contract — not the implementation details, just the "what."

**3. Design** — Figure out *how* to make it happen — the actual algorithms, code structure, and implementation.

Along the way, key principles:
- **Write tests early** (even before implementation), including **extreme values**
- **Reuse existing tools** when possible
- **Refactor** for elegance (DRY, clarity, simplicity, generality)
- Be aware of the **four dimensions**: correctness, efficiency, features, elegance — and make conscious tradeoffs

## Card

`namedtuple` with `patch`

Let's use emoji for suits and string of alphanumerics for ranks.

In [ ]:
suits = '♠♥♦♣'
suits

'♠♥♦♣'

In [ ]:
ranks = '2,3,4,5,6,7,8,9,10,J,Q,K,A'.split(',')
ranks

['2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'A']

Rather than using a string to represent cards and using separate functions to access suits and ranks, I thought it would be better to use the `.` (dot) notation. One way to accomplish this is through `class`, but I wanted to try using `namedtuple`, which allows me to use attribute access and immutability. The ranks and suits from each card should not change. It is more lightweight than `class`.

In [ ]:
Card = namedtuple('Card', ['suit', 'rank'])

c1 = Card('♠', '10')
c1.suit, c1.rank

('♠', '10')

In [ ]:
c1

Card(suit='♠', rank='10')

I wanted the card representation to be more concise, so I used fastcore's `patch` for `__repr__`.

In [ ]:
@patch
def __repr__(self: Card): return self.suit + self.rank

In [ ]:
c2 = Card('♠', 'K')
c2

♠K

Currently, we cannot compare the ranks directly as we are using string comparisons lexicographically. 

In [ ]:
c1.rank > c2.rank, '10' > 'K'

(False, False)

It is convenient to sort hands by rank values when evaluating hands. 

Using `rank_values` mapping, we can convert the string ranks into integers. From 2 to 10, their values are 2 to 10 respectively. Then, from 'J' to 'A', 11 to 14.

In [ ]:
rank_values = {r:i for i,r in enumerate(ranks, start=2)}
rank_values

{'2': 2,
 '3': 3,
 '4': 4,
 '5': 5,
 '6': 6,
 '7': 7,
 '8': 8,
 '9': 9,
 '10': 10,
 'J': 11,
 'Q': 12,
 'K': 13,
 'A': 14}

Although I had `sort_cards` function that sorts cards by their `rank_values` from the highest, it was more convenient to patch `__lt__` and `__eq__` and just use default `sorted` function. When comparing cards, we only care about their ranks, not suits. 

In [ ]:
@patch
def __lt__(self: Card, other: Card): return rank_values[self.rank] < rank_values[other.rank]

In [ ]:
@patch
def __eq__(self: Card, other: Card): return rank_values[self.rank] == rank_values[other.rank]

In [ ]:
deck = [Card(s,r) for s in suits for r in ranks]
sorted(deck,reverse=True)[:5]

[♠A, ♥A, ♦A, ♣A, ♠K]

## Hand

- `evaluate_hand(cards) -> HandRank` — given 7 cards, returns the best 5-card `HandRank`
- `compare_ranks(ranks) -> tuple` — returns the winner rank

A hand has two cards. Community cards have five cards. 

Poker hands:
- royal flush (10)
- straight flush (9, highest_value)
- four of a kind (8, highest_value, kicker)
- full house (7, triple, pair)
- flush (6, five kickers)
- straight (5, highest_value)
- three of a kind (4, triple_value, two kickers)
- two pair (3, high_pair, low_pair, kicker)
- one pair (2, pair_value, three kickers)
- high card (1, highest, second highest, third, fourth, fifth)

### Sample hands

These are some example hands used for examples and tests.

In [ ]:
cards_highcard = [Card(suits[0], ranks[0]), Card(suits[1], ranks[2]), Card(suits[2], ranks[4]),
    Card(suits[3], ranks[6]), Card(suits[0], ranks[8]), Card(suits[1], ranks[10]), Card(suits[2], ranks[12])]
cards_highcard

[♠2, ♥4, ♦6, ♣8, ♠10, ♥Q, ♦A]

In [ ]:
cards_2kind = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[0], ranks[3]), Card(suits[1], ranks[5]), 
    Card(suits[2], ranks[7]), Card(suits[3], ranks[9]), Card(suits[0], ranks[11])]
cards_2kind

[♠2, ♥2, ♠5, ♥7, ♦9, ♣J, ♠K]

In [ ]:
cards_3kind = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[2], ranks[0]), Card(suits[0], ranks[3]), 
    Card(suits[1], ranks[5]), Card(suits[2], ranks[7]), Card(suits[3], ranks[9])]
cards_3kind

[♠2, ♥2, ♦2, ♠5, ♥7, ♦9, ♣J]

In [ ]:
cards_4kind = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[2], ranks[0]), Card(suits[3], ranks[0]),
    Card(suits[0], ranks[4]), Card(suits[1], ranks[7]), Card(suits[2], ranks[10])]
cards_4kind

[♠2, ♥2, ♦2, ♣2, ♠6, ♥9, ♦Q]

In [ ]:
cards_2pair = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[2], ranks[1]), Card(suits[3], ranks[1]),
    Card(suits[0], ranks[4]), Card(suits[1], ranks[6]), Card(suits[2], ranks[8])]
cards_2pair

[♠2, ♥2, ♦3, ♣3, ♠6, ♥8, ♦10]

In [ ]:
cards_fullhouse = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[0], ranks[1]), Card(suits[2], ranks[1]), 
    Card(suits[3], ranks[1]), Card(suits[1], ranks[6]), Card(suits[2], ranks[8])]
cards_fullhouse

[♠2, ♥2, ♠3, ♦3, ♣3, ♥8, ♦10]

In [ ]:
cards_straight = [Card(suits[0], ranks[1]), Card(suits[1], ranks[2]), Card(suits[2], ranks[3]),
    Card(suits[3], ranks[4]), Card(suits[0], ranks[5]), Card(suits[1], ranks[8]), Card(suits[2], ranks[10])]
cards_straight

[♠3, ♥4, ♦5, ♣6, ♠7, ♥10, ♦Q]

In [ ]:
cards_flush = [Card(suits[0], ranks[0]), Card(suits[0], ranks[2]), Card(suits[0], ranks[4]),
    Card(suits[0], ranks[6]), Card(suits[0], ranks[8]), Card(suits[1], ranks[3]), Card(suits[2], ranks[7])]
cards_flush

[♠2, ♠4, ♠6, ♠8, ♠10, ♥5, ♦9]

In [ ]:
cards_straightflush = [Card(suits[0], ranks[1]), Card(suits[0], ranks[2]), Card(suits[0], ranks[3]),
    Card(suits[0], ranks[4]), Card(suits[0], ranks[5]), Card(suits[1], ranks[8]), Card(suits[2], ranks[10])]
cards_straightflush

[♠3, ♠4, ♠5, ♠6, ♠7, ♥10, ♦Q]

In [ ]:
cards_royalflush = [Card(suits[0], ranks[12]), Card(suits[0], ranks[11]), Card(suits[0], ranks[10]), 
    Card(suits[0], ranks[9]), Card(suits[0], ranks[8]), Card(suits[1], ranks[2]), Card(suits[2], ranks[4])]
cards_royalflush

[♠A, ♠K, ♠Q, ♠J, ♠10, ♥4, ♦6]

### Ranks and counts

By getting the counts from the cards, we can figure out `four of a kind`, `full house`, `three of a kind`, `two pair`, and `one pair`. One thing we have to be careful of is because we have seven cards to evaluate, `full house` might have two `three of kind`s, rather than a `three of kind` and a `pair`.

Here is `get_ranks_counts`, which takes a list of cards and returns a tuple of ranks and counts. The output is sorted by the rank counts and `rank_values` for tie breakers. Rank counts are useful for finding what kind of poker ranking hand it is, and ranks are used for the main hand components and kickers.

In [ ]:
# limited: Works with `L[arg]`, but loses the parameter info inside L. Use `types.GenericAlias`
@patch
def __class_getitem__(cls: L, *item): return cls

In [ ]:
def get_ranks_counts(cards: list[Card]) -> L[tuple[int, ...], tuple[int, ...]]:
    """Return ((ranks...), (counts...)) sorted by count then by rank value from cards"""
    items = Counter(rank_values[c.rank] for c in cards).items()
    return L(*sorted(items, key=itemgetter(1, 0), reverse=True)).zip()

In [ ]:
get_ranks_counts(cards_fullhouse), cards_fullhouse

([(3, 2, 10, 8), (3, 2, 1, 1)], [♠2, ♥2, ♠3, ♦3, ♣3, ♥8, ♦10])

In [ ]:
def test_get_ranks_counts():
    test_eq(get_ranks_counts(cards_2kind), ((2, 13, 11, 9, 7, 5), (2, 1, 1, 1, 1, 1)))
    test_eq(get_ranks_counts(cards_3kind), ((2, 11, 9, 7, 5), (3, 1, 1, 1, 1)))
    test_eq(get_ranks_counts(cards_4kind), ((2, 12, 9, 6), (4, 1, 1, 1)))
    test_eq(get_ranks_counts(cards_2pair), ((3, 2, 10, 8, 6), (2, 2, 1, 1, 1)))
    test_eq(get_ranks_counts(cards_fullhouse), ((3, 2, 10, 8), (3, 2, 1, 1)))
    test_eq(get_ranks_counts(cards_highcard), ((14, 12, 10, 8, 6, 4, 2), (1, 1, 1, 1, 1, 1, 1)))
    test_eq(get_ranks_counts(cards_straight), ((12, 10, 7, 6, 5, 4, 3), (1, 1, 1, 1, 1, 1, 1)))

    # Edge case: two three-of-a-kinds (e.g. 222 333 K) — full house should pick higher triple
    cards_2trips = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[2], ranks[0]),
                    Card(suits[0], ranks[1]), Card(suits[1], ranks[1]), Card(suits[2], ranks[1]),
                    Card(suits[3], ranks[11])]
    test_eq(get_ranks_counts(cards_2trips), ((3, 2, 13), (3, 3, 1)))

test_get_ranks_counts()

### flush

As long as the suit count is 5 or more, it is flush because we have seven cards total.

In [ ]:
def is_flush(cards: list[Card]) -> list:
    "If cards have flush, return the `rank_values` from the cards in sorted order. If not, return []"
    suit, count = Counter(c.suit for c in cards).most_common(1)[0]
    return sorted([c for c in cards if c.suit == suit], reverse=True) if count >= 5 else []

In [ ]:
is_flush(cards_flush)

[♠10, ♠8, ♠6, ♠4, ♠2]

In [ ]:
def test_is_flush():
    # Flush: 5 spades, two side cards
    test_eq([str(c) for c in is_flush(cards_flush)], ['♠10', '♠8', '♠6', '♠4', '♠2'])

    # Straight flush
    test_eq([str(c) for c in is_flush(cards_straightflush)], ['♠7', '♠6', '♠5', '♠4', '♠3'])

    # Royal flush
    test_eq([str(c) for c in is_flush(cards_royalflush)], ['♠A', '♠K', '♠Q', '♠J', '♠10'])

    # No flush: suits too spread out
    test_eq(is_flush(cards_fullhouse), [])
    test_eq(is_flush(cards_highcard), [])
    test_eq(is_flush(cards_4kind), [])
    
    # Edge: all 7 cards same suit — returns all 7
    all_spades = [Card('♠', r) for r in 'A,K,Q,J,10,9,8'.split(',')]
    test_eq([str(c) for c in is_flush(all_spades)], ['♠A', '♠K', '♠Q', '♠J', '♠10', '♠9', '♠8'])
    five_hearts = [Card('♥', r) for r in '2,3,4,5,6'.split(',')] + [Card('♠', 'A'), Card('♦', 'K')]
    test_eq([str(c) for c in is_flush(five_hearts)], ['♥6', '♥5', '♥4', '♥3', '♥2'])

test_is_flush()

### straight

Finding straight is trickier because Ace can serve as 1 or 14. Also, there are seven cards, but only five are used. 

In [ ]:
def is_straight(cards: list[Card]) -> int:
    "Return the highest value of the straight if cards contain 5 consecutive ranks, else 0"
    vals = sorted({rank_values[c.rank] for c in cards}, reverse=True)
    if 14 in vals: vals.append(1)
    for i in range(len(vals) - 4):
        if vals[i] - vals[i+4] == 4: return vals[i]
    return 0

In [ ]:
def test_is_straight():
    # Regular straight: 3-4-5-6-7
    test_eq(is_straight(cards_straight), 7)

    # Ace-high straight (10-J-Q-K-A)
    ace_high = [Card('♠','10'), Card('♥','J'), Card('♦','Q'), Card('♣','K'), Card('♠','A'), Card('♥','3'), Card('♦','6')]
    test_eq(is_straight(ace_high), 14)

    # Wheel: A-2-3-4-5 (ace low)
    wheel = [Card('♠','A'), Card('♥','2'), Card('♦','3'), Card('♣','4'), Card('♠','5'), Card('♥','9'), Card('♦','J')]
    test_eq(is_straight(wheel), 5)

    # No straight — gapped
    test_eq(is_straight(cards_2pair), 0)

    # No straight — bunch of duplicates that run together in rank
    dupes = [Card('♠','5'), Card('♥','5'), Card('♦','6'), Card('♣','6'), Card('♠','7'), Card('♥','7'), Card('♦','8')]
    test_eq(is_straight(dupes), 0)

test_is_straight()

### evaluate_hand

Now that we can detect flush and straight, we can update `evaluate_hand`.

In [ ]:
def evaluate_hand(cards: list[Card]) -> tuple:
    """Return hand rank tuple for the best 5-card hand from `cards`.
    Hands ranked from 1 (high card) to 10 (royal flush). Each rank tuple
    starts with the hand category, followed by tiebreaker values (kickers
    or relevant ranks)."""
    ranks, counts = get_ranks_counts(cards)
    c0, c1, *rest = counts
    straight_val = is_straight(cards)
    flush_cards = is_flush(cards)
    sf_val = is_straight(flush_cards)
    if sf_val == 14: return (10,) # royal flush
    elif sf_val: return (9, sf_val) # straight flush
    elif c0 == 4: return (8, ranks[0], ranks[1]) # four of a kind
    elif c0 == 3 and c1 >= 2: return (7, ranks[0], ranks[1]) # full house
    elif flush_cards: return (6, *(rank_values[c.rank] for c in flush_cards)) # flush
    elif straight_val: return (5, straight_val) # straight
    elif c0 == 3: return (4, ranks[0], ranks[1], ranks[2]) # three of a kind
    elif c0 == 2 and c1 == 2: return (3, ranks[0], ranks[1], ranks[2]) # two pair
    elif c0 == 2: return (2, ranks[0], ranks[1], ranks[2], ranks[3]) # one pair
    else: return (1, *ranks) # high card

In [ ]:
def test_evaluate_hand():
    # Royal flush: A♠ K♠ Q♠ J♠ 10♠ → (10,)
    test_eq(evaluate_hand(cards_royalflush), (10,))

    # Straight flush: 3♠ 4♠ 5♠ 6♠ 7♠ → (9, 7)
    test_eq(evaluate_hand(cards_straightflush), (9, 7))

    # Four of a kind: 2222 + Q 9 6 → (8, 2, 12)
    test_eq(evaluate_hand(cards_4kind), (8, 2, 12))

    # Full house: 333 22 + 10 8 → (7, 3, 2)
    test_eq(evaluate_hand(cards_fullhouse), (7, 3, 2))

    # Flush: 5 spades → (6, 10, 8, 6, 4, 2)
    test_eq(evaluate_hand(cards_flush), (6, 10, 8, 6, 4, 2))

    # Straight: 3-4-5-6-7 → (5, 7)
    test_eq(evaluate_hand(cards_straight), (5, 7))

    # Three of a kind: 222 + J 9 7 5 → (4, 2, 11, 9)
    test_eq(evaluate_hand(cards_3kind), (4, 2, 11, 9))
    
    # Two pair: 33 22 + 10 8 6 → (3, 3, 2, 10)
    test_eq(evaluate_hand(cards_2pair), (3, 3, 2, 10))

    # One pair: 22 + K J 9 7 5 → (2, 2, 13, 11, 9)
    test_eq(evaluate_hand(cards_2kind), (2, 2, 13, 11, 9))

    # High card: A Q 10 8 6 4 2 → (1, 14, 12, 10, 8, 6, 4, 2)
    test_eq(evaluate_hand(cards_highcard), (1, 14, 12, 10, 8, 6, 4, 2))

    # ace-high straight without flush should NOT be royal flush
    cards_ace_straight_noflush = [Card('♠','10'), Card('♥','J'), Card('♦','Q'), Card('♣','K'), Card('♠','A'), Card('♥','3'), Card('♦','6')]
    test_eq(evaluate_hand(cards_ace_straight_noflush), (5, 14))

    # straight + flush from different cards should NOT be straight flush
    cards_straight_and_flush = [Card('♠','2'), Card('♠','4'), Card('♠','6'), Card('♠','8'), Card('♠','10'), Card('♥','5'), Card('♥','7')]
    test_eq(evaluate_hand(cards_straight_and_flush), (6, 10, 8, 6, 4, 2))

    # ace-high straight + flush, but flush cards not a straight → flush, not royal flush
    cards_ace_straight_nonstraight_flush = [Card('♠','A'), Card('♠','K'), Card('♠','Q'), Card('♠','J'), Card('♠','5'), Card('♥','10'), Card('♥','3')]
    test_eq(evaluate_hand(cards_ace_straight_nonstraight_flush), (6, 14, 13, 12, 11, 5))

    # straight flush hidden when 6+ cards share a suit and top 5 don't form a straight
    cards_sf_hidden = [Card('♠','2'), Card('♠','3'), Card('♠','4'), Card('♠','5'), Card('♠','6'), Card('♠','8'), Card('♠','K')]
    test_eq(evaluate_hand(cards_sf_hidden), (9, 6))

test_evaluate_hand()

### compare_hands

`compare_hands(hands: List[List[Card]]) -> tuple` — Return the winning hand rank among `hands`

In [ ]:
def compare_hands(hands: list[list[Card]]) -> tuple:
    """Return the winning hand rank among `hands`."""
    return max(evaluate_hand(h) for h in hands)

In [ ]:
evaluate_hand(cards_flush)

(6, 10, 8, 6, 4, 2)

In [ ]:
def test_compare_hands():
    # Clear winner: flush beats one pair
    test_eq(compare_hands([cards_2kind, cards_flush]), (6, 10, 8, 6, 4, 2))

    # Tie: two identical hands → both should win
    test_eq(compare_hands([cards_flush, cards_flush]), (6, 10, 8, 6, 4, 2))

    # Three hands, two tie for best
    test_eq(compare_hands([cards_2kind, cards_flush, cards_royalflush]), (10,))
    
    # Reverse order shouldn't matter
    test_eq(compare_hands([cards_flush, cards_2kind]), (6, 10, 8, 6, 4, 2))

test_compare_hands()

## Deck

In my original plan, I thought of writing a `CardCollection` class as an abstract class and create `Hand`, `Deck`, and `CommunityCards` by inheriting from it as they all consist of cards. However, they don't really have methods that share. To keep things simple, I decided to start implementing `Deck` as a list of `Card`s and see how it feels.

A deck consists of 52 cards. From that, we need to shuffle and withdraw cards so the dealer can deal cards.

In [ ]:
def mk_deck(suits: list = suits, ranks: list = ranks) -> L:
    "Create a deck with given suits and ranks"
    return L(Card(s,r) for s in suits for r in ranks)

In [ ]:
def shuffle_deck(deck: list[Card]) -> list[Card]:
    "Shuffle `deck` in place and return it"
    random.shuffle(deck)
    return deck

In [ ]:
random.seed(42)

deck = mk_deck()
shuffle_deck(deck)[:5]

[♠J, ♥Q, ♥A, ♠5, ♥10]

In [ ]:
def test_deck():
    # Standard deck: 52 cards, 4 suits × 13 ranks
    d = mk_deck()
    test_eq(len(d), 52)

    # No duplicates
    test_eq(len(set(d)), 52)

    # Custom: single suit, 5 ranks → 5 cards
    d5 = mk_deck(suits[0], ranks[:5])
    test_eq(len(d5), 5)
    test_eq(d5, [Card('♠','2'), Card('♠','3'), Card('♠','4'), Card('♠','5'), Card('♠','6')])

    # Two suits, three ranks → 6 cards
    d6 = mk_deck(suits[:2], ranks[:3])
    test_eq(len(d6), 6)
    
    # First card is first suit + first rank
    test_eq(d6[0], Card('♠','2'))
    
test_deck()

In [ ]:
def withdraw_card(deck: list[Card], n: int=1) -> list[Card]:
    "Withdraw `n` cards from `deck` and return them. Modifies `deck` in place."
    if len(deck) < n: raise ValueError(f"Not enough cards: asked for {n}, only {len(deck)} left")
    return [deck.pop() for _ in range(n)]

In [ ]:
def test_withdraw_card():
    # One card: returns it, deck shrinks by one, card was last in deck
    deck = mk_deck()
    last_card = deck[-1]
    got = withdraw_card(deck)
    test_eq(got, [last_card])
    test_eq(len(deck), 51)
    
    # n cards: returned in reverse deck order (pop from end)
    deck = mk_deck()
    expected = list(deck[-3:])[::-1]
    got = withdraw_card(deck, 3)
    test_eq(got, expected)
    test_eq(len(deck), 49)

    # Withdrawn cards are exactly the ones removed — nothing lost or duplicated
    deck = mk_deck()
    full = set(deck)
    got = withdraw_card(deck, 5)
    test_eq(len(set(got) & set(deck)), 0)
    test_eq(set(got) | set(deck), full)

    # n=0: no cards, deck unchanged
    deck = mk_deck()
    test_eq(withdraw_card(deck, 0), [])
    test_eq(len(deck), 52)

    # Withdraw the whole deck: works, deck now empty
    deck = mk_deck()
    test_eq(len(withdraw_card(deck, 52)), 52)
    test_eq(len(deck), 0)

    # Too many: raises, and the deck is untouched
    deck = mk_deck()
    test_fail(lambda: withdraw_card(deck, 53), contains='Not enough cards')
    test_eq(len(deck), 52)

    # Depleted deck: raises even for n=1
    deck = mk_deck()
    withdraw_card(deck, 52)
    test_fail(lambda: withdraw_card(deck), contains='Not enough cards')

test_withdraw_card()

## Player

`Player`class:

- balance: int
- bet: int
- playing: bool
- hand: List[Card] methods:
- take_action(round)
- call(round)
- check(round)
- fold(round)
- raise(round, amount).

In [ ]:
class Action(Enum):
    RAISE = auto()
    CALL = auto()
    CHECK = auto()
    FOLD = auto()

### Strategy

`Player` has a `strategy` attribute, which chooses actions.

In [ ]:
def human(): 
    while True:
        act, *amnt = input("Choose your action").split()
        if (action := getattr(Action, act.upper(), None)):
            return (action, int(*amnt) if amnt else 0)
        print(f"Invalid action: {act}. Try again.")

In [ ]:
def always_call(): return (Action.CALL, 0)

def always_check(): return (Action.CHECK, 0)

def always_fold(): return (Action.FOLD, 0)

def always_raise(n):
    def _raise(): return (Action.RAISE, n)
    _raise.__name__ = f'always_raise({n})'
    return _raise

### player class

In [ ]:
@dataclass
class Player:
    username: str = 'new_user' # username
    balance: int = 0 # total balance
    bet: int = 0 # active bet this round
    playing: bool = True # playing this round or not
    hand: list = field(default_factory=list) # playing hand
    strategy: Callable = always_call # a strategy the player uses to make actions (human or ai)

    def perform_action(self, stage_bet) -> int: return ActionHandler(self, stage_bet).handle()
    
    def reset_bet(self): self.bet = 0

    def reset_playing(self): self.playing = True
    
    def call(self, need): return self._place_bet(need)

    def check(self): return 0

    def fold(self): self.playing = False; return 0

    def raise_(self, need): return self._place_bet(need)
    
    def _place_bet(self, amt):
        self.balance -= amt
        self.bet += amt
        return amt

In [ ]:
@patch
def __repr__(self: Player):
    return (f"Player(username={self.username!r}, balance={self.balance}, bet={self.bet}, "
            f"playing={self.playing}, hand={self.hand}, strategy={self.strategy.__name__})")

In [ ]:
class ActionHandler:
    def __init__(self, player, stage_bet):
        self.player = player
        self.stage_bet = stage_bet
    
    def handle(self):
        action, amount = self.player.strategy()
        print(f'{self.player.username} intends to {action.name.lower()}, stage_bet: {self.stage_bet}, raise_amount: {amount}')
        return getattr(self, '_'+action.name.lower())(amount)

    def _call(self, amount):
        need = self.stage_bet - self.player.bet
        if self.player.balance < need:
            if self.player.strategy != human: return self.player.fold() # AI folds when not enough balance for now..
            print(f"Cannot call. balance ({self.player.balance}) too low to match the stage_bet ({self.stage_bet}). Current bet: {self.player.bet}")
            return self.handle()
        else:
            return self.player.call(need)

    def _check(self, amount): return self.player.check()

    def _fold(self, amount): return self.player.fold()

    def _raise(self, amount):
        new_stage_bet = self.stage_bet + amount
        need = new_stage_bet - self.player.bet
        if need > self.player.balance:
            if self.player.strategy != human: return self.player.fold() # AI folds when not enough balance for now..
            print(f'Cannot raise by {amount}. Your balance: {self.player.balance}, current bet: {self.player.bet}, stage_bet: {self.stage_bet}')
            return self.handle()
        else:
            return self.player.raise_(need)

In [ ]:
stage_bet = 2
player1 = Player('galopy', balance=20, playing=True, strategy=always_raise(1))
need = player1.perform_action(stage_bet)
need, player1

galopy intends to raise, stage_bet: 2, raise_amount: 1


(3,
 Player(username='galopy', balance=17, bet=3, playing=True, hand=[], strategy=always_raise(1)))

In [ ]:
def test_player():
    # call: normal case
    p = Player('t', balance=100, bet=10)
    test_eq(p.call(20), 20)
    test_eq(p.balance, 80)
    test_eq(p.bet, 30)

    # call: need = 0 (already matched)
    p = Player('t', balance=100, bet=10)
    test_eq(p.call(0), 0)
    test_eq(p.balance, 100)
    test_eq(p.bet, 10)

    # check: no state change
    p = Player('t', balance=100, bet=10)
    test_eq(p.check(), 0)
    test_eq(p.balance, 100)
    test_eq(p.bet, 10)

    # fold: playing → False, no money changes
    p = Player('t', balance=100, bet=10)
    test_eq(p.fold(), 0)
    test_eq(p.playing, False)
    test_eq(p.balance, 100)

    # raise_: normal case
    p = Player('t', balance=100, bet=10)
    test_eq(p.raise_(50), 50)
    test_eq(p.balance, 50)
    test_eq(p.bet, 60)

    # call + raise_ share _place_bet: bets accumulate
    p = Player('t', balance=100, bet=10)
    p.call(5); p.raise_(5)
    test_eq(p.balance, 90)
    test_eq(p.bet, 20)
    
    # reset_bet: only bet zeroed, balance untouched
    p = Player('t', balance=100, bet=10)
    p.reset_bet()
    test_eq(p.bet, 0)
    test_eq(p.balance, 100)

test_player()

In [ ]:
def test_action_handler():
    # call: pays the difference between stage_bet and current bet
    p = Player('t', balance=20, bet=0, strategy=always_call)
    test_eq(p.perform_action(2), 2)
    test_eq(p.balance, 18)
    test_eq(p.bet, 2)
    test_eq(p.playing, True)

    # call: partial — player already bet 1, only needs 1 more
    p = Player('t', balance=20, bet=1, strategy=always_call)
    test_eq(p.perform_action(2), 1)
    test_eq(p.balance, 19)
    test_eq(p.bet, 2)

    # check: no state change, returns 0
    p = Player('t', balance=20, bet=0, strategy=always_check)
    test_eq(p.perform_action(2), 0)
    test_eq(p.balance, 20)
    test_eq(p.bet, 0)
    test_eq(p.playing, True)

    # fold: playing → False, no money moves
    p = Player('t', balance=20, bet=0, strategy=always_fold)
    test_eq(p.perform_action(2), 0)
    test_eq(p.playing, False)
    test_eq(p.balance, 20)
    test_eq(p.bet, 0)

    # raise: pays new_stage_bet - current bet
    p = Player('t', balance=20, bet=0, strategy=always_raise(5))
    test_eq(p.perform_action(2), 7)  # 2 + 5 = 7
    test_eq(p.balance, 13)
    test_eq(p.bet, 7)

    # AI folds when it can't afford the call
    p = Player('t', balance=1, bet=0, strategy=always_call)
    test_eq(p.perform_action(2), 0)
    test_eq(p.playing, False)
    test_eq(p.balance, 1)

    # AI folds when it can't afford the raise
    p = Player('t', balance=5, bet=0, strategy=always_raise(10))
    test_eq(p.perform_action(2), 0)
    test_eq(p.playing, False)
    test_eq(p.balance, 5)

test_action_handler()

t intends to call, stage_bet: 2, raise_amount: 0


t intends to call, stage_bet: 2, raise_amount: 0


t intends to check, stage_bet: 2, raise_amount: 0


t intends to fold, stage_bet: 2, raise_amount: 0


t intends to raise, stage_bet: 2, raise_amount: 5


t intends to call, stage_bet: 2, raise_amount: 0


t intends to raise, stage_bet: 2, raise_amount: 10


## Round

Round Management

Attributes:
- stage (pre-flop, flop, turn, river)
- players
- deck
- community_cards

Methods:

- `betting_round(round) -> None`: Manage one round of player actions (pre-flop, flop, turn, or river). Ends when all active players have called or checked.
- `distribute(pot, winners) -> None`: Distribute the pot to the winner(s).
- `start_round(round) -> None`: Orchestrate the whole hand: deal hole cards → blinds → betting round → flop → betting round → turn → betting round → river → betting round → showdown → distribute.

In [ ]:
class Stage(Enum):
    PREFLOP = auto()
    FLOP = auto()
    TURN = auto()
    RIVER = auto()

Let's go through playing a round of poker game!

In [ ]:
class Round:
    def __init__(self, players = None, deck = None, community = None, stage = Stage.PREFLOP, stage_bet = 0, pot = 0):
        self.players = [] if players is None else players
        self.deck = shuffle_deck(mk_deck()) if deck is None else deck
        self.community = [] if community is None else community
        self.stage = stage
        self.stage_bet = stage_bet
        self.pot = pot
        self._idx = 0
        self._raiser = -1
    
    def state(self):
        "Return a string describing the current game situation."
        if self.stage is None: return "Game is over"
        player = self.players[self._idx]
        return f"{player.username}'s turn | Stage: {self.stage.name} | Hand: {player.hand} | \
        Community: {self.community} | Stage bet: {self.stage_bet} | Your bet: {player.bet} | Balance: {player.balance} | Pot: {self.pot}"

In [ ]:
@patch
def _distribute(self: Round, winners) -> None:
    "Distribute the prize pot to the winners"
    print(f"Distributing $ {self.pot} to {winners}")
    for player in winners: player.balance += self.pot // len(winners)

In [ ]:
@patch
def _prep_round(self: Round):
    "Prepare for the round by getting the deck, shuffle, deal hole hands."
    self.deck = shuffle_deck(mk_deck())
    for p in self.players: 
        p.reset_playing()
        p.hand = withdraw_card(self.deck, 2)
    self.community = []
    self.pot = 0
    self.stage = Stage.PREFLOP

In [ ]:
@patch
def _reset_betting_round(self: Round):
    "Reset state for a new betting round."
    self._raiser, self.stage_bet, self._idx = -1, 0, 0
    for p in self.players: p.reset_bet()

In [ ]:
@patch
def _deal_community(self: Round):
    "Deal community cards"
    if self.stage == Stage.FLOP: 
        self.community += withdraw_card(self.deck, 3)
    elif self.stage in (Stage.TURN, Stage.RIVER): 
        self.community += withdraw_card(self.deck)
    print(f"\n--- {self.stage.name} --- Community: {self.community}")

In [ ]:
@patch
def _get_next_player(self: Round) -> Player:
    while True:
        player = self.players[self._idx]
        if player.playing:
            if player.strategy is human:
                bet_amt = yield player
            else:
                bet_amt = player.perform_action(self.stage_bet)
            self.pot += bet_amt
            if player.bet > self.stage_bet:
                self._raiser = self._idx
                self.stage_bet = player.bet
        self._idx = (self._idx + 1) % len(self.players)
        if self._raiser == -1: self._raiser = 0
        if self._idx == self._raiser: break

In [ ]:
@patch
def _showdown(self: Round):
    "Evaluate and distribute."
    players = L.filter(attrgetter('playing'))(self.players)
    winning_score = compare_hands([p.hand + self.community for p in players])
    winners = [p for p in players if evaluate_hand(p.hand + self.community) == winning_score]
    print(f"Winners: {[p.username for p in winners]} with {winning_score}")
    self._distribute(winners)

In [ ]:
@patch
def start_round(self: Round):
    "Deal, start PREFLOP, and auto-advance to first human player."
    self._prep_round()
    self._reset_betting_round()
    for s in Stage:
        self.stage = s
        self._deal_community()
        self._reset_betting_round()
        yield from self._get_next_player()
    self.stage = None
    self._showdown()

In [ ]:
def act(game, player, round):
    if player is None: print("Game is over"); return
    try:
        player = game.send(player.perform_action(round.stage_bet))
    except StopIteration:
        player = None
    return player

In [ ]:
player1 = Player('galopy', balance=20, playing=True, strategy=human)
player2 = Player('jalopy', balance=20, playing=False)
player3 = Player('zalopy', balance=20, playing=True, strategy=human)
players = [player1, player2, player3]
round1 = Round(players = players)
game = round1.start_round()
player = next(game, None)

round1.state()


--- PREFLOP --- Community: []


"galopy's turn | Stage: PREFLOP | Hand: [♥2, ♦Q] |         Community: [] | Stage bet: 0 | Your bet: 0 | Balance: 20 | Pot: 0"

In [ ]:
player = act(game, player, round1)
player

galopy intends to fold, stage_bet: 0, raise_amount: 0


jalopy intends to call, stage_bet: 0, raise_amount: 0


Player(username='zalopy', balance=20, bet=0, playing=True, hand=[♣A, ♣4], strategy=<function human at 0x74e92f42ec00>)

In [ ]:
player = act(game, player, round1)
player

zalopy intends to fold, stage_bet: 0, raise_amount: 0



--- FLOP --- Community: [♦7, ♥A, ♣9]


jalopy intends to call, stage_bet: 0, raise_amount: 0



--- TURN --- Community: [♦7, ♥A, ♣9, ♦5]


jalopy intends to call, stage_bet: 0, raise_amount: 0



--- RIVER --- Community: [♦7, ♥A, ♣9, ♦5, ♠J]


jalopy intends to call, stage_bet: 0, raise_amount: 0


Winners: ['jalopy'] with (2, 9, 14, 11, 8)


Distributing $ 0 to [Player(username='jalopy', balance=20, bet=0, playing=True, hand=[♣8, ♥9], strategy=<function always_call at 0x74e92f42cd60>)]


In [ ]:
player = act(game, player, round1)
player

Game is over


Where can I start writing tests for `Round`? I think we can start with `_distribute`, `_prep_round`, `_reset_betting_round`, `_deal_community`, and `_showdown`. `_get_next_player`, and `start_round` can wait I think. And also do `act` later. What do you think?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good plan — I agree with the ordering. Testing the building blocks first, then deferring the generator-based orchestration (`_get_next_player`, `start_round`, `act`), is the right call. Those three are the most stateful and hardest to reason about, so it helps to have the leaf methods verified first.

A few things to keep in mind as you go:

- **`_distribute`** is the simplest and a natural starting point. You already flagged in your TODO that odd chips vanish (`self.pot // len(winners)` drops the remainder) — write a test that *documents* the current behavior, e.g. pot of 5 split among 2 winners gives each 2. Whether that's correct is a design decision, but the test pins it down either way.

- **`_prep_round`** — good edge cases: verify each player gets exactly 2 cards, the deck has `52 - 2*len(players)` left, and that dealing actually consumed cards from the deck you can inspect. Also check `community`, `pot`, and `stage` all get reset, since `_prep_round` may be called on a Round that has already played a hand.

- **`_deal_community`** — the stage drives the count: FLOP deals 3, TURN/RIVER deal 1. Worth testing each stage, and that cards are appended (not replaced).

- **`_showdown`** — you'll want to construct players with *known* hands plus community cards so you can predict the winner. Also test a tie: two players with equal hand ranks should both win.

One question to consider before writing tests: `_distribute` and `_showdown` both mutate player balances — do you want your test to verify balances before *and* after distribution, or just the outcome?

```json {.usage}
{"model": "deepseek-v4-flash", "prompt_tokens": 50724, "completion_tokens": 1156, "total_tokens": 51880, "cached_tokens": 42112, "reasoning_tokens": 767, "cost": 0.003178392}
```


When writing tests, I don't want to write exhaustive list of tests. My goal is to get a minimum viable product working first then take care of less likely and obscure bugs later.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's a sensible MVP attitude — write just enough to catch the real bugs in the happy path, and skip the exotic edge cases until they actually bite.

For that, I'd suggest this minimal set:

- **`_distribute`**: one normal split (even pot, 1 or 2 winners) and one odd-chips case (since your TODO already flags the remainder behavior as a decision point).
- **`_prep_round`**: one call on a fresh Round — verify each player has 2 cards and `stage == Stage.PREFLOP`. Maybe also call it twice to confirm it resets, since that's the *most likely* bug (state left over from a previous hand).
- **`_reset_betting_round`**: one call — all bets zero, `stage_bet` 0, `_idx` 0. Cheap to test, easy to break.
- **`_deal_community`**: one FLOP call (3 cards) and one TURN call (1 card). Skip RIVER, it's identical logic.
- **`_showdown`**: one hand with a clear winner, and one tie. These matter most for correctness.

That's roughly 8 small tests total — enough to catch the structural bugs without turning it into a chore.

Want to write the `_distribute` test first? It's the smallest one, and it'll let you pin down the odd-chips behavior you already have on your TODO.

```json {.usage}
{"model": "deepseek-v4-flash", "prompt_tokens": 51208, "completion_tokens": 587, "total_tokens": 51795, "cached_tokens": 47872, "reasoning_tokens": 280, "cost": 0.0019524120000000002}
```


It's simple enough test. Can you do this for me?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's a minimal `_distribute` test you can drop into a code message:

```python
def test_distribute():
    # Single winner takes the whole pot
    r = Round(); r.pot = 10
    p = Player(balance=5)
    r._distribute([p])
    test_eq(p.balance, 15)

    # Even split: two winners
    r = Round(); r.pot = 10
    p1, p2 = Player(balance=5), Player(balance=5)
    r._distribute([p1, p2])
    test_eq(p1.balance, 10)
    test_eq(p2.balance, 10)

    # Odd chips: remainder dropped (documents current behavior)
    r = Round(); r.pot = 5
    p1, p2 = Player(balance=0), Player(balance=0)
    r._distribute([p1, p2])
    test_eq(p1.balance, 2)
    test_eq(p2.balance, 2)

test_distribute()
```

One note: the third case pins down the current `pot // len(winners)` behavior where the remainder chip vanishes. If you later decide the remainder should go to the first winner (or stay in the pot for the next hand), you'll change both `_distribute` and this test.

```json {.usage}
{"model": "deepseek-v4-flash", "prompt_tokens": 51583, "completion_tokens": 1294, "total_tokens": 52877, "cached_tokens": 51200, "reasoning_tokens": 1003, "cost": 0.0016991100000000002}
```


In [ ]:
def test_distribute():
    # Single winner takes the whole pot
    r = Round(); r.pot = 10
    p = Player(balance=5)
    r._distribute([p])
    test_eq(p.balance, 15)

    # Even split: two winners
    r = Round(); r.pot = 10
    p1, p2 = Player(balance=5), Player(balance=5)
    r._distribute([p1, p2])
    test_eq(p1.balance, 10)
    test_eq(p2.balance, 10)

    # Odd chips: remainder dropped (documents current behavior)
    r = Round(); r.pot = 5
    p1, p2 = Player(balance=0), Player(balance=0)
    r._distribute([p1, p2])
    test_eq(p1.balance, 2)
    test_eq(p2.balance, 2)

test_distribute()

Distributing $ 10 to [Player(username='new_user', balance=5, bet=0, playing=True, hand=[], strategy=<function always_call at 0x74e92f42cd60>)]


Distributing $ 10 to [Player(username='new_user', balance=5, bet=0, playing=True, hand=[], strategy=<function always_call at 0x74e92f42cd60>), Player(username='new_user', balance=5, bet=0, playing=True, hand=[], strategy=<function always_call at 0x74e92f42cd60>)]


Distributing $ 5 to [Player(username='new_user', balance=0, bet=0, playing=True, hand=[], strategy=<function always_call at 0x74e92f42cd60>), Player(username='new_user', balance=0, bet=0, playing=True, hand=[], strategy=<function always_call at 0x74e92f42cd60>)]


LGTM

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()